Environment

In [ ]:
import sys

libs = {
  # "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
  "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}
for lib in libs.values():
  sys.path.append(lib)

Import libraries

In [ ]:
import os
import numpy as np
import scipy as sp
import dask.array as da
import tensorflow as tf

from silx.io.dictdump import h5todict
from functions import *
from utils import *

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Path to database
path_to_dtb = "./../database/"
# Path to FOM and ROM solutions
path_to_sol = "/home/zanardi/Workspace/AirDatabase/ThermoChemModels/StS/cr_argon/testcases/runs/1d_shock_unsteady_cc/fom/vel1/output/"
# Path to saving folder
path_to_save = "./output/"
# Flow model
flow_model = "EU_NLTE"
# Load solution from pre-saved file
load_from_file = True
# Domain limits
xlim = [0.0, 0.2]
tlim = [0.0, 2.5e-4]
# Solution snapshot to be plotted
itime = -1
# Show plots
showing = False
# Error upper bound for clipping
err_max = 10.0

In [ ]:
os.makedirs(path_to_save, exist_ok=True)

In [ ]:
sol = read_sol(
  dt=5e-8,
  path_to_sol=path_to_sol,
  path_to_dtb=path_to_dtb,
  savefile=path_to_save + "/sol.p",
  flow_model=flow_model,
  load_from_file=load_from_file,
  save=True
)

In [ ]:
tf.nest.map_structure(np.shape, sol)

In [ ]:
i = sol["p"] > sol["p"][0,0]

In [ ]:
w = sol["w"][i,:]
wln = np.log(w)
# Pareto
wln_m = np.mean(wln, axis=0, keepdims=True)
wln_s = np.std(wln, axis=0, keepdims=True)
wln_n = (wln-wln_m)/wln_s

In [ ]:
# from sklearn.utils.extmath import randomized_svd

In [ ]:
# U, s, Vh = randomized_svd(wln_n, n_components=15, n_iter=50, power_iteration_normalizer='QR', random_state=0)

In [ ]:
yd = da.from_array(wln_n.T, chunks="auto")
u, s, vh = da.linalg.svd_compressed(yd, k=15)
s = s.compute()
u = u.compute()

In [ ]:
cs = 1.0 / np.sum(s**2)
cs *= np.cumsum(s**2)

In [ ]:
# Number of principal components
eps = 1e-5
nb_pc = np.where(cs > 1-eps)[0][0] + 1
nb_pc

In [ ]:
cs

In [ ]:
plot_cumenergy(
  cs[:10],
  # figname=paths["figs"] + "/cum_en.png",
  save=False,
  show=True
)

In [ ]:
def reconstruct(w, phi, m, s):
  # Transform
  y = np.log(w)
  # Normalize
  y = (y - m) / s
  # Encode
  z = y @ phi
  # Decode
  yh = z @ phi.T
  # Denormalize
  yh = yh * s + m
  # Transform
  yh = np.exp(yh)
  return yh

In [ ]:
z = np.random.rand(5)
y = np.exp(phi @ z)
zr = phi.T @ np.log(y)
z, zr

In [ ]:
phi = u[:,:5]
wr = reconstruct(w, phi, wln_m, wln_s)

In [ ]:
err = 100*np.abs(wr-w)/(w+1e-15)
err = np.amax(err, axis=-1)
i = np.flip(np.argsort(err))
i

In [ ]:
x = np.arange(34)
# i = np.random.choice(len(w))
# print(i)
# i = 10
for j in i[:10]:
  plt.scatter(x=x, y=w[j])
  plt.scatter(x=x, y=wr[j], marker="x")
  plt.yscale("log")
  plt.show()
  plt.close()

In [ ]:
stop

In [ ]:
w = sol["w"][i,:]
wln = np.log(w)
# Pareto
wln_m = np.mean(wln, axis=0, keepdims=True)
wln_s = np.std(wln, axis=0, keepdims=True)
wln_n = (wln-wln_m)/wln_s

Define inputs

In [ ]:
nb_workers = 16
# Plotting
saving = True
showing = True
# Variable to decompose
# variable = {
#     "name": "na",
#     "norm": "$\\tilde{n}_{o}$",
#     "text": r"$n_{o}$ [m$^{-3}$]"
# }
variable = {
    "name": "pa",
    "norm": "$\\tilde{p}_{o}$",
    "text": r"$p_{o}$ [Pa]"
}
# variable = {
#     "name": "T",
#     "norm": "$\\tilde{T}$",
#     "text": r"$T$ [K]"
# }

Define paths

In [ ]:
paths = {
    "figs": f"./figs/{variable['name']}/",
    "tensors": f"./tensors/{variable['name']}/",
    "sols": [
        "./../datagen/runs/samples/data/train/sols.hdf5",
        "./../datagen/runs/samples/data/valid/sols.hdf5"
    ]
}
for k in ("figs", "tensors"):
    os.makedirs(paths[k], exist_ok=True)

Read solutions

In [ ]:
sols = []
for p in paths["sols"]: 
    sols.extend(list(h5todict(p).values()))

Plot solutions

In [ ]:
def get_xy(sol, pscale=False, minmax=False):
    T = sol["y"][:,-1:]
    na = sol["y"][:,:1]
    pa = na * const.UKB * T
    # Input
    x = sol["t"]
    if pscale:
        x = x * pa
    # Output
    if (variable['name'] == "T"):
        y = T
    elif (variable['name'] == "pa"):
        y = pa
    else:
        y = na
    if minmax:
        y = (y - np.amin(y)) / (np.amax(y) - np.amin(y))
    return x.squeeze(), y.squeeze()

> Function of $t$

In [ ]:
plot_traj(
    xy=[get_xy(sol, pscale=False, minmax=True) for sol in sols],
    labels=['$t$ [s]', variable['norm']],
    scales=['log', 'linear'],
    figname=paths["figs"] + f"/data.png",
    save=saving,
    show=showing
)

> Function of $t\cdot p_\text{o}$

In [ ]:
plot_traj(
    xy=[get_xy(sol, pscale=True, minmax=True) for sol in sols],
    labels=[r'$t\cdot p_{o}(t)$ [s Pa]', variable['norm']],
    scales=['log', 'linear'],
    figname=paths["figs"] + f"/data_pscale.png",
    save=saving,
    show=showing
)

Perfoming SVD
> Interpolate solutions onto the same grid

In [ ]:
def interpolate(t, sol):
    x, y = get_xy(sol, pscale=True, minmax=False)
    # Extend domain
    x = np.concatenate([x, x[-1:]*1e2])
    y = np.concatenate([y, y[-1:]])
    # Interpolate
    return sp.interpolate.interp1d(x, y, kind='cubic')(t)

In [ ]:
tp_lim = (-7, 0)
tp = np.logspace(*tp_lim, 1000)

In [ ]:
y = [interpolate(tp, sol) for sol in sols]
y = np.vstack(y).T

> SVD decomposition

In [ ]:
m = np.mean(y, axis=0, keepdims=True)
s = np.std(y, axis=0, keepdims=True)
yn = (y - m) / s

In [ ]:
yd = da.from_array(yn, chunks="auto")
u, s, vh = da.linalg.svd_compressed(yd, k=200)
s = s.compute()
u = u.compute()

> Singular values decay

In [ ]:
cs = 1.0 / np.sum(s**2)
cs *= np.cumsum(s**2)

In [ ]:
# Number of principal components
eps = 1e-5
nb_pc = np.where(cs > 1-eps)[0][0] + 1
nb_pc

In [ ]:
plot_cumenergy(
  cs[:10],
  figname=paths["figs"] + "/cum_en.png",
  save=saving,
  show=showing
)

> Extract modes

In [ ]:
phi = u[:,:nb_pc]
np.savetxt(paths["tensors"] + "/phi.txt", phi, fmt='%.10e')
np.savetxt(paths["tensors"] + "/tp.txt", tp, fmt='%.10e')

In [ ]:
path = paths["figs"] + "/modes/"
os.makedirs(path, exist_ok=True)
for i in range(nb_pc):
    mode_nb = str(i+1)
    plot_2d(
        x=tp,
        y_true=phi[:,i],
        labels=[r'$t\cdot p_{o}(t)$ [s Pa]', r"$\phi_{{%s}}$" % mode_nb],
        scales=["log", "linear"],
        filename=path + f"/phi_{mode_nb.zfill(2)}.png",
        save=saving,
        show=showing
    )

> Test reconstruction

In [ ]:
def reconstruct(y):
    m = np.mean(y)
    s = np.std(y)
    yn = (y - m) / s
    w = ols(phi, yn)
    yr = (phi @ w) * s + m
    return yr

In [ ]:
path = paths["figs"] + "/rec/"
os.makedirs(path, exist_ok=True)

In [ ]:
cases = np.random.randint(low=0, high=len(sols), size=10)

In [ ]:
for icase in cases:
    yi = interpolate(tp, sols[icase])
    yr = reconstruct(yi)
    plot_2d(
        x=tp,
        y_true=yi,
        y_pred=yr,
        labels=[r'$t\cdot p_{o}(t)$ [s Pa]', variable['text']],
        scales=["log", "linear"],
        filename=path + f"/{str(icase).zfill(6)}.png",
        save=saving,
        show=showing
    )